### Setup

In [ ]:
# custom imports
import nekt
import os                           
from dotenv import load_dotenv      
from typing import Optional 
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# get Nekt data access token        
load_dotenv()
nekt.data_access_token = os.getenv("NEKT_DATA_ACCESS_TOKEN")

### Functions

In [ ]:
# auxiliary functions
def extract_nekt_table(table_name: str, layer_name: str) -> DataFrame:
    return nekt.load_table(layer_name=layer_name, table_name=table_name)

def save_nekt_table(
    df: DataFrame, 
    layer_name: str, 
    table_name: str,
    folder_name: Optional[str] = None 
):
    nekt.save_table(
        df=df,
        layer_name=layer_name,
        table_name=table_name,
        folder_name=folder_name
    )

def display(df: DataFrame, limit=10):
    return df.limit(limit).toPandas()

### Extraction

In [ ]:
# clickup
df_bronze_clickup_users        = extract_nekt_table("users"        , "Bronze")
df_bronze_clickup_spaces       = extract_nekt_table("spaces"       , "Bronze")
df_bronze_clickup_time_entries = extract_nekt_table("time_entries" , "Bronze")

In [ ]:
# conta azul
df_bronze_contaazul_account_payables    = extract_nekt_table("despesas"        , "Bronze")
df_bronze_contaazul_account_receivables = extract_nekt_table("receitas"        , "Bronze")
df_bronze_contaazul_installments        = extract_nekt_table("parcelas"        , "Bronze")
df_bronze_contaazul_sales               = extract_nekt_table("time_entries"    , "Bronze")
# df_bronze_contaazul_clientes = load_nekt_table("time_entries" , "Bronze")
# df_bronze_contaazul_vendas = load_nekt_table("time_entries" , "Bronze")
# df_bronze_contaazul_pessoas = load_nekt_table("time_entries" , "Bronze")

### Tranformations

In [ ]:
# clickup
# df_silver_clickup_users = df_bronze_clickup_users.select(
#       F.col("id").cast("integer")     .alias("user_id")
#     , F.col("username")               .alias("user_name")
# ).filter(
#     F.col("user_name").isNotNull()
# ).dropDuplicates(
#     ["user_id"]
# )

# df_silver_clickup_spaces = df_bronze_clickup_spaces.select(
#       F.col("id").cast("long")        .alias("space_id")
#     , F.col("name")                   .alias("space_name")
# ).filter(
#     F.col("space_name").isNotNull()
# ).dropDuplicates(
#     ["space_id"]
# )

# df_silver_clickup_time_entries = df_bronze_clickup_time_entries.select(
#       F.col("id").cast("long")                        .alias("interval_id")
#     , F.col("wid").cast("integer")                    .alias("team_id")
#     , F.col("user.id").cast("integer")                .alias("user_id")
#     , F.col("user.username")                          .alias("user_name")
#     , F.col("task_location.space_id").cast("long")    .alias("space_id")
#     , F.col("task.id").alias("task_id")
#     , F.col("start").cast("long")                     .alias("interval_date_start_ms")
#     , F.to_timestamp(F.col("start") / 1000)           .alias("interval_date_start_iso")
#     , F.col("end").cast("long")                       .alias("interval_date_end_ms")
#     , F.to_timestamp(F.col("end") / 1000)             .alias("interval_date_end_iso")
#     , F.col("at").cast("long")                        .alias("interval_date_added_ms")
#     , F.to_timestamp(F.col("at") / 1000)              .alias("interval_date_added_iso")
# ).filter(
#       F.col("interval_id").isNotNull()
#     & F.col("user_id").isNotNull()
#     & F.col("interval_date_start_ms").isNotNull()
#     & F.col("interval_date_end_ms").isNotNull()
#     & (F.col("interval_date_end_ms") > F.col("interval_date_start_ms"))  # validating business rule
# ).dropDuplicates(
#     ["interval_id"]
# )

In [ ]:
# conta azul
## contas a pagar
df_bronze_contaazul_account_payables = df_bronze_contaazul_account_payables.select(
      F.col("id")
    , F.col("descricao")
    , F.col("data_vencimento")
    , F.col("status")
    , F.col("total")
    , F.col("nao_pago")
    , F.col("pago")
    , F.col("data_criacao")
    , F.col("data_alteracao")
    # , F.col("categoria_principal_id")
    # , F.col("categoria_principal_nome")
    # , F.col("fornecedor_id")
    # , F.col("fornecedor_nome")
    # , F.col("_loaded_at")
)

## contas a receber
df_bronze_contaazul_account_receivables = df_bronze_contaazul_account_receivables.select(
      F.col("id")
    , F.col("descricao")
    , F.col("data_vencimento")
    , F.col("status")
    , F.col("total")
    , F.col("nao_pago")
    , F.col("pago")
    , F.col("data_criacao")
    , F.col("data_alteracao")
    # , F.col("categoria_principal_id")
    # , F.col("categoria_principal_nome")
    # , F.col("fornecedor_id")
    # , F.col("fornecedor_nome")
    # , F.current_timestamp().alias("loaded_at")
)

## parcelas
df_bronze_contaazul_installments = df_bronze_contaazul_installments.select(
      F.col("id")                       .alias("parcela_id")
    , F.col("status")                   .alias("parcela_status")
    , F.col("evento.condicao_pagamento").alias("condicao_pagamento")
    , F.col("referencia")
    , F.col("evento.agendado")          .alias("agendado")
    , F.col("evento.tipo")              .alias("tipo_evento")
    , F.col("")                         .alias("")
)

## baixa de parcelas
df_bronze_contaazul_settled_installments = df_bronze_contaazul_installments.select(
      F.col("id")                                       .alias("parcela_id")
    , F.explode("baixas")                               .alias("baixa")).filter(
        F.size("baixas") > 0 
    ).select(
          F.col("parcela_id")
        , F.col("baixa.id")                             .alias("baixa_id")
        , F.col("baixa.versao")                         .alias("baixa_versao")
        , F.col("baixa.data_pagamento")                 .alias("baixa_data_pagamento") 
        , F.col("baixa.id_reconciliacao")               .alias("baixa_id_reconciliacao")
        , F.col("baixa.id_solicitacao_cobranca")        .alias("baixa_solicitacao_cobranca")
        , F.col("baixa.observacao")                     .alias("baixa_observacao")
        , F.col("baixa.metodo_pagamento")               .alias("baixa_metodo_pagamento")
        , F.col("baixa.origem")                         .alias("baixa_origem")
        , F.col("baixa.id_recibo_digital")              .alias("baixa_id_recibo_digital")
        , F.col("baixa.tipo_evento_financeiro")         .alias("baixa_tipo_evento_financeiro")
        , F.col("baixa.nsu")                            .alias("baixa_nsu")
        , F.col("baixa.id_referencia")                  .alias("baixa_id_referencia")
        , F.col("baixa.atualizado_em")                  .alias("baixa_atualizado_em")
        , F.col("baixa.valor_composicao.desconto")      .alias("baixa_desconto")
        , F.col("baixa.valor_composicao.juros")         .alias("baixa_juros")
        , F.col("baixa.valor_composicao.multa")         .alias("baixa_multa")
        , F.col("baixa.valor_composicao.taxa")          .alias("baixa_taxa")
        , F.col("baixa.valor_composicao.valor_bruto")   .alias("baixa_valor_bruto")
        , F.col("baixa.valor_composicao.valor_liquido") .alias("baixa_valor_liquido")
        , F.current_timestamp()                         .alias("loaded_at")
    )

### Saving

In [ ]:
# clickup
# save_nekt_table(df_silver_users         , "Silver", "users",        folder_name="studio_61_clickup")
# save_nekt_table(df_silver_spaces        , "Silver", "spaces",       folder_name="studio_61_clickup")
# save_nekt_table(df_silver_time_entries  , "Silver", "time_entries", folder_name="studio_61_clickup")

In [ ]:
# conta azul
# save_nekt_table(df_silver_users         , "Silver", "users",        folder_name="studio_61_clickup")
# save_nekt_table(df_silver_spaces        , "Silver", "spaces",       folder_name="studio_61_clickup")
# save_nekt_table(df_silver_time_entries  , "Silver", "time_entries", folder_name="studio_61_clickup")